In [1]:
import os
import gc
import glob
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from scipy.signal import savgol_filter
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings("ignore")

SEED = 42
N_JOBS = max(1, min(os.cpu_count() or 2, 8))

DATA_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")

if not DATA_DIR.exists():
    DATA_DIR = Path("/kaggle/input/rogii-wellbore-geology-prediction")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

print("DATA_DIR:", DATA_DIR)
print("N_JOBS:", N_JOBS)

DATA_DIR: /kaggle/input/competitions/rogii-wellbore-geology-prediction
N_JOBS: 4


In [2]:
def get_well_ids(folder):
    files = sorted(folder.glob("*__horizontal_well.csv"))
    return [f.name.replace("__horizontal_well.csv", "") for f in files]

def hfile(folder, wid):
    return folder / f"{wid}__horizontal_well.csv"

def tfile(folder, wid):
    # Competition data seems to usually use this name:
    p = folder / f"{wid}__typewell.csv"
    if p.exists():
        return p
    
    # Fallback for alternate naming:
    matches = sorted(folder.glob(f"{wid}__typewell*.csv"))
    if matches:
        return matches[0]
    
    raise FileNotFoundError(f"No typewell file for {wid}")

train_wells = get_well_ids(TRAIN_DIR)
test_wells = get_well_ids(TEST_DIR)

print("Train wells:", len(train_wells))
print("Test wells:", len(test_wells))

Train wells: 773
Test wells: 3


In [3]:
def np_roll_mean(x, w):
    x = np.asarray(x, dtype=np.float32)
    s = pd.Series(x)
    return s.rolling(w, center=True, min_periods=1).mean().to_numpy(np.float32)

def np_roll_std(x, w):
    x = np.asarray(x, dtype=np.float32)
    s = pd.Series(x)
    return s.rolling(w, center=True, min_periods=2).std().fillna(0).to_numpy(np.float32)

def safe_interp(x, xp, fp):
    return np.interp(x, xp, fp).astype(np.float32)

def robust_slope(x, y, tail=None):
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    
    if tail is not None and len(x) > tail:
        x = x[-tail:]
        y = y[-tail:]
    
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return 0.0
    
    if np.std(x[mask]) < 1e-6:
        return 0.0
    
    return float(np.polyfit(x[mask], y[mask], 1)[0])

In [4]:
def fast_multiscale_ncc(kgr, ktvt, hgr, windows=(8, 15, 25), stride=3):
    """
    Fast normalized cross-correlation between:
    - known horizontal GR before PS
    - hidden/evaluation GR after PS

    Returns predicted TVT-like anchors from known prefix similarity.
    """
    kgr = np.asarray(kgr, dtype=np.float32)
    ktvt = np.asarray(ktvt, dtype=np.float32)
    hgr = np.asarray(hgr, dtype=np.float32)
    
    n_eval = len(hgr)
    outputs = []
    scores = []
    
    for half_w in windows:
        win = 2 * half_w + 1
        
        if len(kgr) < win + 1 or n_eval == 0:
            outputs.append(np.full(n_eval, ktvt[-1], dtype=np.float32))
            scores.append(np.zeros(n_eval, dtype=np.float32))
            continue
        
        kg = np_roll_mean(kgr, 5)
        hg = np_roll_mean(hgr, 5)
        
        starts = np.arange(0, len(kg) - win + 1, stride, dtype=np.int32)
        
        if len(starts) == 0:
            outputs.append(np.full(n_eval, ktvt[-1], dtype=np.float32))
            scores.append(np.zeros(n_eval, dtype=np.float32))
            continue
        
        # Candidate known windows: shape = [num_candidates, win]
        C = kg[starts[:, None] + np.arange(win)[None, :]]
        C = C.astype(np.float32)
        C = (C - C.mean(axis=1, keepdims=True)) / (C.std(axis=1, keepdims=True) + 1e-6)
        
        # Eval windows: shape = [n_eval, win]
        hp = np.pad(hg, half_w, mode="edge")
        H = hp[np.arange(n_eval)[:, None] + np.arange(win)[None, :]]
        H = H.astype(np.float32)
        H = (H - H.mean(axis=1, keepdims=True)) / (H.std(axis=1, keepdims=True) + 1e-6)
        
        # NCC: [n_eval, num_candidates]
        ncc = H @ C.T / win
        
        best = np.argmax(ncc, axis=1)
        best_score = np.max(ncc, axis=1).astype(np.float32)
        best_tvt = ktvt[np.clip(starts[best] + half_w, 0, len(ktvt) - 1)]
        
        outputs.append(best_tvt.astype(np.float32))
        scores.append(best_score)
    
    outputs = np.stack(outputs, axis=1)
    scores = np.stack(scores, axis=1)
    
    weights = np.exp(3.0 * scores)
    weights = weights / (weights.sum(axis=1, keepdims=True) + 1e-9)
    
    ens = (outputs * weights).sum(axis=1).astype(np.float32)
    
    result = {
        "ncc8_tvt": outputs[:, 0],
        "ncc15_tvt": outputs[:, 1],
        "ncc25_tvt": outputs[:, 2],
        "ncc8_score": scores[:, 0],
        "ncc15_score": scores[:, 1],
        "ncc25_score": scores[:, 2],
        "ncc_ens_tvt": ens,
    }
    
    return result

In [5]:
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]

# These are GR/TWT offsets used for fast typewell-GR comparison features.
ANCHOR_OFFSETS = np.array(
    [-100, -75, -50, -35, -25, -15, -10, -5, 0, 5, 10, 15, 25, 35, 50, 75, 100],
    dtype=np.float32
)

# Spatial imputation settings copied conceptually from rogii-dual-pipeline-blend.
# The key idea: do NOT use raw ANCC/ASTNU/... train-only columns directly.
# Instead, learn formation surfaces from training wells, then impute equivalent
# formation surfaces for BOTH train and test wells using X/Y location.
PLANE_K = 10
DENSE_SPW = 60
DENSE_K = 20

_FI = None  # FormationPlaneKNN global imputer
_DI = None  # DenseANCCImputer global imputer


def seg_b_well(ktvt, kz, form_col):
    """
    Calibrate imputed formation surface to known TVT_input prefix.

    If formation depth ~= Z + TVT - bias, then:
        bias ~= TVT + Z - formation_depth

    Returns several prefix-bias estimates, matching the example notebook's method.
    """
    ktvt = np.asarray(ktvt, dtype=np.float32)
    kz = np.asarray(kz, dtype=np.float32)
    form_col = np.asarray(form_col, dtype=np.float32)

    bv = ktvt + kz - form_col
    bv = bv[np.isfinite(bv)]

    if len(bv) == 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    n = len(bv)
    b_full = float(np.nanmedian(bv))
    b_late = float(np.nanmedian(bv[max(0, n - 50):])) if n >= 5 else b_full

    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.nanmedian(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.nanmedian(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full

    w = np.exp(0.02 * np.arange(n, dtype=np.float32))
    w = w / (w.sum() + 1e-12)
    b_wls = float(np.dot(w, bv))

    return b_full, b_early, b_mid, b_late, b_wls


class FormationPlaneKNN:
    """
    Learns local formation surfaces from training wells and imputes formation
    depths at arbitrary X/Y query points.

    For train feature generation, self_wid is excluded from neighbors so the
    model does not get its own exact train-only formation columns back.
    """
    def __init__(self, well_ids, data_dir, formations=FORMATION_COLS):
        self.formations = formations
        rows = []

        for wid in well_ids:
            try:
                df = pd.read_csv(
                    data_dir / f"{wid}__horizontal_well.csv",
                    usecols=["X", "Y"] + formations,
                ).dropna()
            except Exception:
                continue

            if len(df) == 0:
                continue

            row = {
                "wid": wid,
                "x": float(df["X"].median()),
                "y": float(df["Y"].median()),
            }

            for c in formations:
                row[f"{c}_m"] = float(df[c].median())

            rows.append(row)

        self.df = pd.DataFrame(rows)

        if len(self.df) == 0:
            raise RuntimeError("FormationPlaneKNN could not find usable formation columns in training data.")

        self.wmap = {w: i for i, w in enumerate(self.df["wid"].values)}
        xy = self.df[["x", "y"]].to_numpy(np.float64)
        self.scale = np.where(xy.std(axis=0) < 1e-3, 1.0, xy.std(axis=0))
        self.tree = cKDTree(xy / self.scale)

        self.xa = self.df["x"].to_numpy(np.float64)
        self.ya = self.df["y"].to_numpy(np.float64)
        self.fa = self.df[[f"{c}_m" for c in formations]].to_numpy(np.float64)
        self.global_mean = self.fa.mean(axis=0).astype(np.float32)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        xy_q = np.atleast_2d(np.asarray(xy_q, dtype=np.float64))
        q = xy_q / self.scale

        # Fetch extra neighbors so excluding self_wid still leaves enough candidates.
        nf = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=nf, workers=-1)

        if nf == 1:
            dist = dist[:, None]
            idx = idx[:, None]

        if self_wid in self.wmap:
            dist = np.where(idx == self.wmap[self_wid], np.inf, dist)

        kk = min(k, nf)
        take = min(kk - 1, nf - 1)
        order = np.argpartition(dist, take, axis=1)[:, :kk]
        dk = np.take_along_axis(dist, order, axis=1)
        ik = np.take_along_axis(idx, order, axis=1)

        valid = np.isfinite(dk)
        weights = np.where(valid, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)

        xn = self.xa[ik]
        yn = self.ya[ik]
        fn = self.fa[ik]

        # Weighted local plane: formation_depth = a*X + b*Y + c
        wx = weights * xn
        wy = weights * yn

        A = np.zeros((len(q), 3, 3), dtype=np.float64)
        A[:, 0, 0] = (wx * xn).sum(axis=1)
        A[:, 0, 1] = (wx * yn).sum(axis=1)
        A[:, 0, 2] = wx.sum(axis=1)
        A[:, 1, 0] = A[:, 0, 1]
        A[:, 1, 1] = (wy * yn).sum(axis=1)
        A[:, 1, 2] = wy.sum(axis=1)
        A[:, 2, 0] = A[:, 0, 2]
        A[:, 2, 1] = A[:, 1, 2]
        A[:, 2, 2] = weights.sum(axis=1)

        A[:, 0, 0] += 1e-9
        A[:, 1, 1] += 1e-9
        A[:, 2, 2] += 1e-9

        rhs = np.stack(
            [
                (wx[:, :, None] * fn).sum(axis=1),
                (wy[:, :, None] * fn).sum(axis=1),
                (weights[:, :, None] * fn).sum(axis=1),
            ],
            axis=1,
        )

        try:
            coef = np.linalg.solve(A, rhs)
        except Exception:
            coef = np.zeros((len(q), 3, len(self.formations)), dtype=np.float64)
            for r in range(len(q)):
                try:
                    coef[r] = np.linalg.pinv(A[r]) @ rhs[r]
                except Exception:
                    pass

        xq = xy_q[:, 0]
        yq = xy_q[:, 1]
        pred = xq[:, None] * coef[:, 0, :] + yq[:, None] * coef[:, 1, :] + coef[:, 2, :]
        pred = pred.astype(np.float32)

        no_valid = ~valid.any(axis=1)
        if no_valid.any():
            pred[no_valid] = self.global_mean

        min_dist = np.where(valid, dk, np.inf).min(axis=1).astype(np.float32)
        return pred, min_dist


class DenseANCCImputer:
    """
    Dense sampled KNN imputer for ANCC only, copied conceptually from the example.
    This gives an additional high-resolution spatial prior for the most useful formation.
    """
    def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
        xs, ys, ancc, wids = [], [], [], []

        for wid in well_ids:
            try:
                df = pd.read_csv(
                    data_dir / f"{wid}__horizontal_well.csv",
                    usecols=["X", "Y", "ANCC"],
                ).dropna()
            except Exception:
                continue

            if len(df) == 0:
                continue

            ix = np.linspace(0, len(df) - 1, min(spw, len(df)), dtype=int)
            s = df.iloc[ix]
            xs.append(s["X"].values)
            ys.append(s["Y"].values)
            ancc.append(s["ANCC"].values)
            wids.extend([wid] * len(s))

        if not xs:
            raise RuntimeError("DenseANCCImputer could not find usable ANCC data in training data.")

        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)]).astype(np.float64)
        self.ancc = np.concatenate(ancc).astype(np.float32)
        self.wids = np.asarray(wids)
        self.scale = np.where(self.xy.std(axis=0) < 1e-3, 1.0, self.xy.std(axis=0))
        self.tree = cKDTree(self.xy / self.scale)
        self.global_mean = float(np.mean(self.ancc))

    def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=5000):
        xy_q = np.atleast_2d(np.asarray(xy_q, dtype=np.float64))
        q = xy_q / self.scale
        nf = min(nfetch, len(self.ancc))

        dist, idx = self.tree.query(q, k=nf, workers=-1)

        if nf == 1:
            dist = dist[:, None]
            idx = idx[:, None]

        if self_wid is not None:
            dist = np.where(self.wids[idx] == self_wid, np.inf, dist)

        kk = min(k, nf)
        take = min(kk - 1, nf - 1)
        order = np.argpartition(dist, take, axis=1)[:, :kk]
        dk = np.take_along_axis(dist, order, axis=1)
        ik = np.take_along_axis(idx, order, axis=1)

        valid = np.isfinite(dk)
        weights = np.where(valid, 1.0 / (dk + 1e-3), 0.0).astype(np.float64)
        sw = weights.sum(axis=1)
        safe = np.where(sw < 1e-9, 1.0, sw)

        vals = self.ancc[ik]
        pred = (vals * weights).sum(axis=1) / safe
        pred = np.where(sw < 1e-9, self.global_mean, pred)

        var = ((vals - pred[:, None]) ** 2 * weights).sum(axis=1) / safe
        std = np.sqrt(np.maximum(var, 0.0))
        min_dist = np.where(valid, dk, np.inf).min(axis=1)

        return pred.astype(np.float32), std.astype(np.float32), min_dist.astype(np.float32)


def init_formation_imputers(train_wids=None, train_dir=None, force=False):
    """Initialize global formation imputers once before threaded feature generation."""
    global _FI, _DI

    if _FI is not None and _DI is not None and not force:
        return

    if train_wids is None:
        train_wids = train_wells
    if train_dir is None:
        train_dir = TRAIN_DIR

    print("Initializing formation imputers from training wells...")
    _FI = FormationPlaneKNN(train_wids, train_dir)
    _DI = DenseANCCImputer(train_wids, train_dir)
    print(f"FormationPlaneKNN wells used: {len(_FI.df):,}")
    print(f"DenseANCCImputer samples used: {len(_DI.ancc):,}")


def add_imputed_formation_features(feats, wid, is_train, kn, ev, ktvt, last_tvt, ncc):
    """
    Adds formation features that exist for both train and test:
    - Impute formation surfaces from training-well spatial KNN/planes.
    - Exclude the current well during train feature generation to avoid leakage.
    - Calibrate imputed surfaces against the known TVT_input prefix.
    """
    global _FI, _DI

    if _FI is None or _DI is None:
        init_formation_imputers(train_wells, TRAIN_DIR)

    swid = wid if is_train else None

    xy_ev = ev[["X", "Y"]].to_numpy(np.float64)
    xy_kn = kn[["X", "Y"]].to_numpy(np.float64)

    z_kn = kn["Z"].to_numpy(np.float32)
    z_ev = ev["Z"].to_numpy(np.float32)

    form_ev, knn_dist = _FI.impute(xy_ev, self_wid=swid)
    form_kn, _ = _FI.impute(xy_kn, self_wid=swid)

    form_tvts = []
    n = len(ev)

    feats["spatial_knn_dist"] = knn_dist.astype(np.float32)

    for j, fn in enumerate(FORMATION_COLS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, form_kn[:, j])

        tvt_f = (-z_ev + form_ev[:, j] + b_full).astype(np.float32)
        tvt_fw = (-z_ev + form_ev[:, j] + b_wls).astype(np.float32)
        tvt_f50 = (-z_ev + form_ev[:, j] + b_late).astype(np.float32)

        pred_kn = (-z_kn + form_kn[:, j] + b_full).astype(np.float32)
        frm_rmse = float(np.sqrt(np.nanmean((ktvt - pred_kn) ** 2))) if len(pred_kn) else 0.0

        # Absolute calibrated TVT candidates.
        feats[f"tvtF_{fn}"] = tvt_f
        feats[f"tvtFw_{fn}"] = tvt_fw
        feats[f"tvtF50_{fn}"] = tvt_f50

        # Delta versions usually work better with target = TVT - last_known_tvt.
        feats[f"tvtF_{fn}_d"] = (tvt_f - last_tvt).astype(np.float32)
        feats[f"tvtFw_{fn}_d"] = (tvt_fw - last_tvt).astype(np.float32)
        feats[f"tvtF50_{fn}_d"] = (tvt_f50 - last_tvt).astype(np.float32)

        # Bias/calibration diagnostics.
        feats[f"bw_{fn}"] = np.full(n, b_full, dtype=np.float32)
        feats[f"bww_{fn}"] = np.full(n, b_wls, dtype=np.float32)
        feats[f"bw50_{fn}"] = np.full(n, b_late, dtype=np.float32)
        feats[f"bw_early_{fn}"] = np.full(n, b_early, dtype=np.float32)
        feats[f"bw_mid_{fn}"] = np.full(n, b_mid, dtype=np.float32)
        feats[f"frm_rmse_{fn}"] = np.full(n, frm_rmse, dtype=np.float32)

        # Raw imputed surface diagnostics are safe because they are imputed for train/test.
        feats[f"imp_{fn}_surf"] = form_ev[:, j].astype(np.float32)
        feats[f"imp_{fn}_surf_minus_z"] = (form_ev[:, j] - z_ev).astype(np.float32)

        form_tvts.append(tvt_f)

    fs = np.stack(form_tvts, axis=1)
    feats["form_mean_d"] = (fs.mean(axis=1) - last_tvt).astype(np.float32)
    feats["form_std_d"] = fs.std(axis=1).astype(np.float32)
    feats["form_rng_d"] = (fs.max(axis=1) - fs.min(axis=1)).astype(np.float32)

    # Dense ANCC imputer features.
    dense_ev, dense_std, dense_dist = _DI.impute(xy_ev, self_wid=swid)
    dense_kn, dense_std_kn, _ = _DI.impute(xy_kn, self_wid=swid)

    b_full, b_early, b_mid, b_late, b_wls = seg_b_well(ktvt, z_kn, dense_kn)
    tvt_dense = (-z_ev + dense_ev + b_full).astype(np.float32)
    tvt_densew = (-z_ev + dense_ev + b_wls).astype(np.float32)
    tvt_dense50 = (-z_ev + dense_ev + b_late).astype(np.float32)

    pred_dense_kn = (-z_kn + dense_kn + b_full).astype(np.float32)
    dense_rmse = float(np.sqrt(np.nanmean((ktvt - pred_dense_kn) ** 2))) if len(pred_dense_kn) else 0.0
    dense_bias = float(np.nanmean(ktvt + z_kn - dense_kn)) if len(dense_kn) else 0.0
    dense_nb_std = float(np.nanmean(dense_std_kn)) if len(dense_std_kn) else 0.0

    feats["dense_ancc"] = dense_ev.astype(np.float32)
    feats["dense_std"] = dense_std.astype(np.float32)
    feats["dense_dist"] = dense_dist.astype(np.float32)
    feats["tvt_dense"] = tvt_dense
    feats["tvt_densew"] = tvt_densew
    feats["tvt_dense50"] = tvt_dense50
    feats["tvt_dense_d"] = (tvt_dense - last_tvt).astype(np.float32)
    feats["tvt_densew_d"] = (tvt_densew - last_tvt).astype(np.float32)
    feats["tvt_dense50_d"] = (tvt_dense50 - last_tvt).astype(np.float32)
    feats["dense_rmse"] = np.full(n, dense_rmse, dtype=np.float32)
    feats["dense_bias"] = np.full(n, dense_bias, dtype=np.float32)
    feats["dense_nb_std"] = np.full(n, dense_nb_std, dtype=np.float32)

    # Cross-signal differences: cheap and often useful.
    feats["ncc_vs_spatial_ancc"] = (ncc["ncc_ens_tvt"] - feats["tvtF_ANCC"]).astype(np.float32)
    feats["ncc_vs_dense"] = (ncc["ncc_ens_tvt"] - tvt_dense).astype(np.float32)
    feats["spatial_ancc_vs_dense"] = (feats["tvtF_ANCC"] - tvt_dense).astype(np.float32)

    return feats


def build_one_well_fast(wid, folder, is_train=True):
    try:
        hw = pd.read_csv(hfile(folder, wid))
        tw = pd.read_csv(tfile(folder, wid)).sort_values("TVT")
    except Exception as e:
        print(f"[WARN] failed reading {wid}: {e}")
        return None

    if is_train and "TVT" not in hw.columns:
        return None

    if "TVT_input" not in hw.columns:
        return None

    known_mask = hw["TVT_input"].notna()
    eval_mask = hw["TVT_input"].isna()

    if eval_mask.sum() == 0 or known_mask.sum() < 10:
        return None

    kn = hw.loc[known_mask].copy()
    ev = hw.loc[eval_mask].copy()

    if is_train:
        if "TVT" not in ev.columns or ev["TVT"].isna().all():
            return None

    # Typewell arrays.
    tw = tw.dropna(subset=["TVT", "GR"])
    tw_tvt = tw["TVT"].to_numpy(np.float32)
    tw_gr = tw["GR"].to_numpy(np.float32)

    if len(tw_tvt) < 5:
        return None

    # Fill GR once.
    gr_full = (
        hw["GR"]
        .astype(float)
        .interpolate(limit_direction="both")
        .fillna(float(np.nanmean(tw_gr)))
        .to_numpy(np.float32)
    )

    eval_idx = ev.index.to_numpy()
    known_idx = kn.index.to_numpy()

    kgr = gr_full[known_idx]
    hgr = gr_full[eval_idx]

    ktvt = kn["TVT_input"].to_numpy(np.float32)
    kmd = kn["MD"].to_numpy(np.float32)
    kz = kn["Z"].to_numpy(np.float32)

    lk = kn.iloc[-1]
    last_tvt = np.float32(lk["TVT_input"])
    last_md = np.float32(lk["MD"])
    last_x = np.float32(lk["X"])
    last_y = np.float32(lk["Y"])
    last_z = np.float32(lk["Z"])

    n = len(ev)

    ev_md = ev["MD"].to_numpy(np.float32)
    ev_x = ev["X"].to_numpy(np.float32)
    ev_y = ev["Y"].to_numpy(np.float32)
    ev_z = ev["Z"].to_numpy(np.float32)

    md_since = ev_md - last_md

    # Simple slope baselines.
    slope_all = robust_slope(kmd, ktvt)
    slope_50 = robust_slope(kmd, ktvt, tail=50)
    slope_150 = robust_slope(kmd, ktvt, tail=150)
    slope_z = robust_slope(kz, ktvt)

    lin_all = last_tvt + slope_all * md_since
    lin_50 = last_tvt + slope_50 * md_since
    lin_150 = last_tvt + slope_150 * md_since
    lin_z = last_tvt + slope_z * (ev_z - last_z)

    # Fast NCC features.
    ncc = fast_multiscale_ncc(kgr, ktvt, hgr)

    # Typewell GR at baseline TVT predictions.
    tw_gr_last = np.float32(np.interp(last_tvt, tw_tvt, tw_gr))
    tw_gr_lin_all = safe_interp(lin_all, tw_tvt, tw_gr)
    tw_gr_lin_50 = safe_interp(lin_50, tw_tvt, tw_gr)
    tw_gr_ncc = safe_interp(ncc["ncc_ens_tvt"], tw_tvt, tw_gr)

    # Fast offset features around last known TVT.
    offset_feats = {}
    for off in ANCHOR_OFFSETS:
        ref_tvt = last_tvt + off
        offset_feats[f"gr_minus_tw_last_off_{int(off)}"] = (
            hgr - np.float32(np.interp(ref_tvt, tw_tvt, tw_gr))
        ).astype(np.float32)

    # Offset features around slope baseline.
    for off in [-50, -25, -10, 0, 10, 25, 50]:
        off = np.float32(off)
        offset_feats[f"gr_minus_tw_lin50_off_{int(off)}"] = (
            hgr - safe_interp(lin_50 + off, tw_tvt, tw_gr)
        ).astype(np.float32)

    # Full-well derivative features.
    md_diff = hw["MD"].diff().replace(0, np.nan)
    dzdmd_full = (hw["Z"].diff() / md_diff).to_numpy(np.float32)
    dxdmd_full = (hw["X"].diff() / md_diff).to_numpy(np.float32)
    dydmd_full = (hw["Y"].diff() / md_diff).to_numpy(np.float32)

    # Rolling GR features for full well, selected only at eval rows.
    grm5 = np_roll_mean(gr_full, 5)[eval_idx]
    grm21 = np_roll_mean(gr_full, 21)[eval_idx]
    grm51 = np_roll_mean(gr_full, 51)[eval_idx]
    grs21 = np_roll_std(gr_full, 21)[eval_idx]
    grs51 = np_roll_std(gr_full, 51)[eval_idx]

    # Lag/lead features.
    gr_series = pd.Series(gr_full)

    feats = {
        "well": wid,
        "id": [f"{wid}_{i}" for i in eval_idx],

        "last_known_tvt": np.full(n, last_tvt, dtype=np.float32),
        "last_known_md": np.full(n, last_md, dtype=np.float32),

        "md": ev_md,
        "md_since": md_since.astype(np.float32),
        "frac": (np.arange(n) / max(n - 1, 1)).astype(np.float32),
        "frac2": (np.arange(n) / max(n - 1, 1)).astype(np.float32) ** 2,

        "x": ev_x,
        "y": ev_y,
        "z": ev_z,
        "dx": (ev_x - last_x).astype(np.float32),
        "dy": (ev_y - last_y).astype(np.float32),
        "dz": (ev_z - last_z).astype(np.float32),
        "dxy": np.sqrt((ev_x - last_x) ** 2 + (ev_y - last_y) ** 2).astype(np.float32),

        "dzdmd": dzdmd_full[eval_idx],
        "dxdmd": dxdmd_full[eval_idx],
        "dydmd": dydmd_full[eval_idx],

        "gr": hgr,
        "grm5": grm5.astype(np.float32),
        "grm21": grm21.astype(np.float32),
        "grm51": grm51.astype(np.float32),
        "grs21": grs21.astype(np.float32),
        "grs51": grs51.astype(np.float32),
        "gr_d1": np.nan_to_num(np.diff(gr_full, prepend=gr_full[0]))[eval_idx].astype(np.float32),
        "gr_d2": np.nan_to_num(np.diff(np.diff(gr_full, prepend=gr_full[0]), prepend=0))[eval_idx].astype(np.float32),

        "gr_lag1": gr_series.shift(1).bfill().to_numpy(np.float32)[eval_idx],
        "gr_lag5": gr_series.shift(5).bfill().to_numpy(np.float32)[eval_idx],
        "gr_lag15": gr_series.shift(15).bfill().to_numpy(np.float32)[eval_idx],
        "gr_lead1": gr_series.shift(-1).ffill().to_numpy(np.float32)[eval_idx],
        "gr_lead5": gr_series.shift(-5).ffill().to_numpy(np.float32)[eval_idx],
        "gr_lead15": gr_series.shift(-15).ffill().to_numpy(np.float32)[eval_idx],

        "slope_all": np.full(n, slope_all, dtype=np.float32),
        "slope_50": np.full(n, slope_50, dtype=np.float32),
        "slope_150": np.full(n, slope_150, dtype=np.float32),
        "slope_z": np.full(n, slope_z, dtype=np.float32),

        "lin_all_d": (lin_all - last_tvt).astype(np.float32),
        "lin_50_d": (lin_50 - last_tvt).astype(np.float32),
        "lin_150_d": (lin_150 - last_tvt).astype(np.float32),
        "lin_z_d": (lin_z - last_tvt).astype(np.float32),

        "tw_gr_last": np.full(n, tw_gr_last, dtype=np.float32),
        "tw_gr_lin_all": tw_gr_lin_all,
        "tw_gr_lin_50": tw_gr_lin_50,
        "tw_gr_ncc": tw_gr_ncc,

        "gr_minus_tw_last": (hgr - tw_gr_last).astype(np.float32),
        "gr_minus_tw_lin_all": (hgr - tw_gr_lin_all).astype(np.float32),
        "gr_minus_tw_lin_50": (hgr - tw_gr_lin_50).astype(np.float32),
        "gr_minus_tw_ncc": (hgr - tw_gr_ncc).astype(np.float32),

        "ncc8_d": (ncc["ncc8_tvt"] - last_tvt).astype(np.float32),
        "ncc15_d": (ncc["ncc15_tvt"] - last_tvt).astype(np.float32),
        "ncc25_d": (ncc["ncc25_tvt"] - last_tvt).astype(np.float32),
        "ncc_ens_d": (ncc["ncc_ens_tvt"] - last_tvt).astype(np.float32),
        "ncc8_score": ncc["ncc8_score"],
        "ncc15_score": ncc["ncc15_score"],
        "ncc25_score": ncc["ncc25_score"],

        "known_len": np.full(n, len(kn), dtype=np.float32),
        "eval_len": np.full(n, len(ev), dtype=np.float32),
        "known_tvt_std": np.full(n, np.std(ktvt), dtype=np.float32),
        "known_tvt_range": np.full(n, np.ptp(ktvt), dtype=np.float32),
        "tw_tvt_range": np.full(n, np.ptp(tw_tvt), dtype=np.float32),
        "tw_gr_mean": np.full(n, np.mean(tw_gr), dtype=np.float32),
        "tw_gr_std": np.full(n, np.std(tw_gr), dtype=np.float32),

        **offset_feats,
    }

    # Add imputed/calibrated formation features that are available for BOTH train and test.
    # This replaces the old raw train-only ANCC/ASTNU/... feature block.
    feats = add_imputed_formation_features(feats, wid, is_train, kn, ev, ktvt, last_tvt, ncc)

    out = pd.DataFrame(feats)

    if is_train:
        out["target"] = (ev["TVT"].to_numpy(np.float32) - last_tvt).astype(np.float32)

    return out


In [6]:
def build_dataset_fast(wells, folder, is_train=True, cache_path=None):
    # Required for imputed formation features. Initialize once before Parallel threads start.
    if _FI is None or _DI is None:
        init_formation_imputers(train_wells, TRAIN_DIR)

    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists():
            print(f"Loading cached features: {cache_path}")
            return pd.read_parquet(cache_path)
    
    t0 = time.time()
    
    results = Parallel(n_jobs=N_JOBS, prefer="threads", verbose=10)(
        delayed(build_one_well_fast)(wid, folder, is_train)
        for wid in wells
    )
    
    parts = [r for r in results if r is not None and len(r) > 0]
    
    if not parts:
        return pd.DataFrame()
    
    df = pd.concat(parts, ignore_index=True)
    
    # Downcast floats aggressively.
    for c in df.columns:
        if df[c].dtype == "float64":
            df[c] = df[c].astype(np.float32)
    
    elapsed = time.time() - t0
    print(f"Built {len(df):,} rows from {len(parts):,} wells in {elapsed/60:.2f} min")
    
    if cache_path is not None:
        df.to_parquet(cache_path, index=False)
        print(f"Saved cache: {cache_path}")
    
    return df

In [7]:
# IMPORTANT: use new cache names because the feature schema changed.
# Old caches like train_fast_features.parquet do NOT include imputed formation features.
init_formation_imputers(train_wells, TRAIN_DIR)

train_df = build_dataset_fast(
    train_wells,
    TRAIN_DIR,
    is_train=True,
    cache_path="/kaggle/working/train_fast_features_imputed_formations.parquet"
)

test_df = build_dataset_fast(
    test_wells,
    TEST_DIR,
    is_train=False,
    cache_path="/kaggle/working/test_fast_features_imputed_formations.parquet"
)

print(train_df.shape)
print(test_df.shape)

formation_feature_examples = [c for c in train_df.columns if c.startswith(("tvtF_", "imp_", "dense_", "form_"))]
print("Imputed formation feature count:", len(formation_feature_examples))
print("Example imputed formation features:", formation_feature_examples[:30])

display(train_df.head())


Initializing formation imputers from training wells...
FormationPlaneKNN wells used: 765
DenseANCCImputer samples used: 45,960


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   13.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   19.9s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:   31.6s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:   45.9s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   58.6s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:  1.3min
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:  1.6min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:  1.9min
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:  2.4min
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:  2.7min
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:  3.2min
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:  3.6min
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:  4.2min
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:  4.6min
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:  5.2min
[Para

Built 3,783,989 rows from 773 wells in 23.30 min
Saved cache: /kaggle/working/train_fast_features_imputed_formations.parquet


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   3 out of   3 | elapsed:    4.9s finished


Built 14,151 rows from 3 wells in 0.08 min
Saved cache: /kaggle/working/test_fast_features_imputed_formations.parquet
(3783989, 190)
(14151, 189)
Imputed formation feature count: 33
Example imputed formation features: ['tvtF_ANCC', 'tvtF_ANCC_d', 'imp_ANCC_surf', 'imp_ANCC_surf_minus_z', 'tvtF_ASTNU', 'tvtF_ASTNU_d', 'imp_ASTNU_surf', 'imp_ASTNU_surf_minus_z', 'tvtF_ASTNL', 'tvtF_ASTNL_d', 'imp_ASTNL_surf', 'imp_ASTNL_surf_minus_z', 'tvtF_EGFDU', 'tvtF_EGFDU_d', 'imp_EGFDU_surf', 'imp_EGFDU_surf_minus_z', 'tvtF_EGFDL', 'tvtF_EGFDL_d', 'imp_EGFDL_surf', 'imp_EGFDL_surf_minus_z', 'tvtF_BUDA', 'tvtF_BUDA_d', 'imp_BUDA_surf', 'imp_BUDA_surf_minus_z', 'form_mean_d', 'form_std_d', 'form_rng_d', 'dense_ancc', 'dense_std', 'dense_dist']


,well,id,last_known_tvt,last_known_md,md,md_since,frac,frac2,x,y,...,tvt_dense_d,tvt_densew_d,tvt_dense50_d,dense_rmse,dense_bias,dense_nb_std,ncc_vs_spatial_ancc,ncc_vs_dense,spatial_ancc_vs_dense,target
0,000d7d20,000d7d20_1442,11747.370117,12908.0,12909.0,1.0,0.000000,0.000000e+00,2983537.0,1070212.75,...,-1.191406,-0.429688,-0.300781,4.758359,11388.692383,18.73012,-64.087891,-60.757812,3.330078,0.009766
1,000d7d20,000d7d20_1443,11747.370117,12908.0,12910.0,2.0,0.000261,6.799380e-08,2983537.0,1070213.75,...,-1.195312,-0.433594,-0.304688,4.758359,11388.692383,18.73012,-68.293945,-64.947266,3.346680,0.019531
2,000d7d20,000d7d20_1444,11747.370117,12908.0,12911.0,3.0,0.000522,2.719752e-07,2983537.0,1070214.75,...,-0.028320,0.733398,0.862305,4.758359,11388.692383,18.73012,-68.411133,-66.217773,2.193359,0.030273
3,000d7d20,000d7d20_1445,11747.370117,12908.0,12912.0,4.0,0.000782,6.119441e-07,2983537.0,1070215.75,...,-0.030273,0.731445,0.860352,4.758359,11388.692383,18.73012,-67.818359,-65.610352,2.208008,0.030273
4,000d7d20,000d7d20_1446,11747.370117,12908.0,12913.0,5.0,0.001043,1.087901e-06,2983537.0,1070216.75,...,-0.032227,0.729492,0.858398,4.758359,11388.692383,18.73012,-72.241211,-70.017578,2.223633,0.040039


In [8]:
DROP_COLS = {
    "id",
    "well",
    "target",
}

# IMPORTANT:
# Build features only from columns that exist in BOTH train_df and test_df.
# This prevents train-only columns such as ANCC/ASTNU/BUDA formation-surface features
# from being used during training and then crashing during prediction.
COMMON_COLS = sorted(set(train_df.columns).intersection(set(test_df.columns)))

FEATURES = [
    c for c in COMMON_COLS
    if c not in DROP_COLS
    and train_df[c].dtype != "O"
    and test_df[c].dtype != "O"
]

TRAIN_ONLY_COLS = sorted(set(train_df.columns) - set(test_df.columns))
TEST_ONLY_COLS = sorted(set(test_df.columns) - set(train_df.columns))

print("Num shared features:", len(FEATURES))
print("Train-only columns excluded:", len(TRAIN_ONLY_COLS))
print("Test-only columns ignored:", len(TEST_ONLY_COLS))

if TRAIN_ONLY_COLS:
    print("Example train-only excluded columns:", TRAIN_ONLY_COLS[:40])

IMPUTED_FORMATION_FEATURES = [
    c for c in FEATURES
    if c.startswith(("tvtF_", "tvtFw_", "tvtF50_", "imp_", "dense_", "form_", "bw_", "bww_", "bw50_", "frm_rmse_"))
]
RAW_TRAIN_ONLY_FORMATION_FEATURES = [
    c for c in FEATURES
    if c in FORMATION_COLS or c.endswith(("_surf", "_surf_minus_z", "_struct_d", "_bias")) and not c.startswith("imp_")
]

print("Imputed formation features used:", len(IMPUTED_FORMATION_FEATURES))
print("Example imputed formation features used:", IMPUTED_FORMATION_FEATURES[:30])
# assert len(RAW_TRAIN_ONLY_FORMATION_FEATURES) == 0, RAW_TRAIN_ONLY_FORMATION_FEATURES[:20]

# Safety check: this should always be empty.
missing_in_test = [c for c in FEATURES if c not in test_df.columns]
assert len(missing_in_test) == 0, f"FEATURES still contains columns missing in test_df: {missing_in_test[:20]}"

# LightGBM GPU likes dense float32 data. Avoid repeatedly slicing Pandas objects
# more than necessary during CV.
X = (
    train_df[FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype(np.float32)
)

y = train_df["target"].values.astype(np.float32)
groups = train_df["well"].values
last_known = train_df["last_known_tvt"].values.astype(np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num wells:", train_df["well"].nunique())

# Toggle these depending on whether you are iterating or doing a final run.
FAST_MODE = False          # True = faster dev run, False = stronger run
USE_GPU = True             # Uses Kaggle GPU if LightGBM build supports it
TRY_CUDA_FIRST = False     # Usually keep False on Kaggle; OpenCL "gpu" is more likely to work

if FAST_MODE:
    N_SPLITS = 3
    N_ESTIMATORS = 1500
    LEARNING_RATE = 0.04
    NUM_LEAVES = 63
    EARLY_STOPPING_ROUNDS = 100
else:
    N_SPLITS = 5
    N_ESTIMATORS = 5000
    LEARNING_RATE = 0.025
    NUM_LEAVES = 127
    EARLY_STOPPING_ROUNDS = 250

def lightgbm_gpu_available(device_type="gpu"):
    """
    Quick runtime test. Kaggle environments can change, and some LightGBM builds
    expose CPU only. This prevents the notebook from crashing after a long setup.
    """
    if not USE_GPU:
        return False

    try:
        x_small = np.random.RandomState(SEED).randn(2000, 16).astype(np.float32)
        y_small = np.random.RandomState(SEED + 1).randn(2000).astype(np.float32)

        test_model = LGBMRegressor(
            objective="regression",
            n_estimators=5,
            learning_rate=0.1,
            num_leaves=31,
            max_bin=63,
            device_type=device_type,
            gpu_use_dp=False,
            verbosity=-1,
            n_jobs=2,
            random_state=SEED,
        )
        test_model.fit(x_small, y_small)
        return True
    except Exception as e:
        print(f"LightGBM device_type='{device_type}' failed, falling back if possible.")
        print("Reason:", str(e).split("\n")[0])
        return False

# Pick the best working device.
DEVICE_TYPE = "cpu"

if USE_GPU:
    candidate_devices = ["cuda", "gpu"] if TRY_CUDA_FIRST else ["gpu", "cuda"]

    for dev in candidate_devices:
        if lightgbm_gpu_available(dev):
            DEVICE_TYPE = dev
            break

print("Using LightGBM device_type:", DEVICE_TYPE)

Num shared features: 187
Train-only columns excluded: 1
Test-only columns ignored: 0
Example train-only excluded columns: ['target']
Imputed formation features used: 93
Example imputed formation features used: ['bw50_ANCC', 'bw50_ASTNL', 'bw50_ASTNU', 'bw50_BUDA', 'bw50_EGFDL', 'bw50_EGFDU', 'bw_ANCC', 'bw_ASTNL', 'bw_ASTNU', 'bw_BUDA', 'bw_EGFDL', 'bw_EGFDU', 'bw_early_ANCC', 'bw_early_ASTNL', 'bw_early_ASTNU', 'bw_early_BUDA', 'bw_early_EGFDL', 'bw_early_EGFDU', 'bw_mid_ANCC', 'bw_mid_ASTNL', 'bw_mid_ASTNU', 'bw_mid_BUDA', 'bw_mid_EGFDL', 'bw_mid_EGFDU', 'bww_ANCC', 'bww_ASTNL', 'bww_ASTNU', 'bww_BUDA', 'bww_EGFDL', 'bww_EGFDU']
X shape: (3783989, 187)
y shape: (3783989,)
Num wells: 773


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Using LightGBM device_type: gpu


In [9]:
params = dict(
    objective="regression",
    metric="rmse",
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    num_leaves=NUM_LEAVES,
    max_depth=-1,
    min_child_samples=25,
    subsample=0.90,
    subsample_freq=1,
    colsample_bytree=0.90,
    reg_alpha=0.03,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

if DEVICE_TYPE in ("gpu", "cuda"):
    # Important GPU settings:
    # - max_bin=63 usually speeds up GPU histogram construction a lot.
    # - gpu_use_dp=False uses 32-bit accumulation. Faster, usually fine for RMSE competitions.
    # - force_col_wise/force_row_wise should NOT be set for GPU mode.
    params.update(
        dict(
            device_type=DEVICE_TYPE,
            max_bin=63,
            gpu_use_dp=False,
        )
    )
else:
    # CPU fallback. This avoids LightGBM's row-wise/col-wise auto-test overhead.
    params.update(
        dict(
            force_col_wise=True,
            max_bin=255,
        )
    )

print("Training params:")
for k, v in params.items():
    if k in ["device_type", "max_bin", "learning_rate", "n_estimators", "num_leaves", "gpu_use_dp", "force_col_wise"]:
        print(f"  {k}: {v}")

gkf = GroupKFold(n_splits=N_SPLITS)

oof = np.zeros(len(train_df), dtype=np.float32)
models = []
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups), 1):
    print(f"\nFold {fold}/{N_SPLITS}")

    fold_params = params.copy()
    fold_params["random_state"] = SEED + fold

    model = LGBMRegressor(**fold_params)

    model.fit(
        X.iloc[tr_idx],
        y[tr_idx],
        eval_set=[(X.iloc[va_idx], y[va_idx])],
        eval_metric="rmse",
        callbacks=[
            early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
            log_evaluation(250),
        ],
    )

    best_iter = model.best_iteration_ or N_ESTIMATORS
    pred = model.predict(X.iloc[va_idx], num_iteration=best_iter).astype(np.float32)
    oof[va_idx] = pred

    rmse_delta = np.sqrt(mean_squared_error(y[va_idx], pred))

    true_abs = last_known[va_idx] + y[va_idx]
    pred_abs = last_known[va_idx] + pred
    rmse_abs = np.sqrt(mean_squared_error(true_abs, pred_abs))

    fold_scores.append(rmse_abs)

    print("Best iter:", best_iter)
    print("Delta RMSE:", rmse_delta)
    print("Absolute TVT RMSE:", rmse_abs)

    models.append(model)
    gc.collect()

overall_abs_rmse = np.sqrt(mean_squared_error(
    last_known + y,
    last_known + oof,
))

print("\nFold absolute RMSEs:", fold_scores)
print("OOF absolute RMSE:", overall_abs_rmse)

Training params:
  n_estimators: 5000
  learning_rate: 0.025
  num_leaves: 127
  device_type: gpu
  max_bin: 63
  gpu_use_dp: False

Fold 1/5
[250]	valid_0's rmse: 10.0181
[500]	valid_0's rmse: 9.96409
[750]	valid_0's rmse: 9.94088
[1000]	valid_0's rmse: 9.93663
[1250]	valid_0's rmse: 9.93625
Best iter: 1155
Delta RMSE: 9.934560664944815
Absolute TVT RMSE: 9.934560664944815

Fold 2/5
[250]	valid_0's rmse: 12.5953
Best iter: 59
Delta RMSE: 12.485048431070336
Absolute TVT RMSE: 12.485049042152816

Fold 3/5
[250]	valid_0's rmse: 10.4746
[500]	valid_0's rmse: 10.4285
[750]	valid_0's rmse: 10.4225
Best iter: 619
Delta RMSE: 10.42155472064558
Absolute TVT RMSE: 10.42155472064558

Fold 4/5
[250]	valid_0's rmse: 12.6947
[500]	valid_0's rmse: 12.6793
[750]	valid_0's rmse: 12.6787
Best iter: 693
Delta RMSE: 12.675429742814996
Absolute TVT RMSE: 12.675429742814996

Fold 5/5
[250]	valid_0's rmse: 12.5971
[500]	valid_0's rmse: 12.5406
[750]	valid_0's rmse: 12.5316
Best iter: 720
Delta RMSE: 12.5299

In [10]:
# Match training preprocessing exactly.
X_test = test_df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

test_pred_delta = np.zeros(len(test_df), dtype=np.float32)

for model in models:
    best_iter = model.best_iteration_ or params.get("n_estimators", None)
    test_pred_delta += model.predict(X_test, num_iteration=best_iter).astype(np.float32) / len(models)

test_df["pred_tvt"] = test_df["last_known_tvt"].values.astype(np.float32) + test_pred_delta

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

sub = sample[["id"]].merge(
    test_df[["id", "pred_tvt"]],
    on="id",
    how="left"
)

sub["tvt"] = sub["pred_tvt"]
sub = sub.drop(columns=["pred_tvt"])

# Fallback if somehow missing.
fallback = float(train_df["last_known_tvt"].mean() + train_df["target"].mean())
sub["tvt"] = sub["tvt"].fillna(fallback)

print("Missing predictions:", sub["tvt"].isna().sum())
sub.head()

Missing predictions: 0


,id,tvt
0,000d7d20_1442,11747.722656
1,000d7d20_1443,11747.720703
2,000d7d20_1444,11747.723633
3,000d7d20_1445,11747.714844
4,000d7d20_1446,11747.728516


In [11]:
sub.to_csv("submission.csv", index=False)